# Untied BERT Bi-Encoder — Pairwise Similarity Tester

Enter two sentences and get a similarity score in [0, 1].

**How scoring works:**  
- Sentence A is encoded with the **anchor encoder**  
- Sentence B is encoded with the **positive encoder**  
- Both embeddings are L2-normalized → dot product = cosine similarity ∈ [-1, 1]  
- Final score is scaled to [0, 1] via `(cosine + 1) / 2`

A score near **1.0** means "B is a likely follow-up / effect of A";  
a score near **0.0** means "B is unrelated to A".

In [1]:
import sys
from pathlib import Path

# Make sure the project root and src/ are on the path
REPO_ROOT = Path("__file__").resolve().parent.parent
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "src"))

import torch
from transformers import AutoTokenizer
from biencoder.model import BiEncoder

print("Imports OK")

/mnt/localssd/internship-causal-embedding-merge-followup/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


## 1 — Pick a checkpoint

Each dataset has its own fine-tuned checkpoint.  
Set `DATASET` to whichever model you want to probe:

In [8]:
# ── CHANGE ME ─────────────────────────────────────────────────────────────────
DATASET   = "followupqg"          # aep_causal | followupqg | multiwoz_v24 | qrecc | workflow
VARIANT   = ""                    # "" for in-batch baseline, "_no_inbatch" for hard-only variant
# ──────────────────────────────────────────────────────────────────────────────

RESULTS_DIR = REPO_ROOT / "finetune_eval" / "results"
CKPT_PATH   = RESULTS_DIR / (DATASET + VARIANT) / "checkpoint_best.pt"

assert CKPT_PATH.exists(), f"Checkpoint not found: {CKPT_PATH}"
print(f"Using checkpoint: {CKPT_PATH}")

Using checkpoint: /mnt/localssd/new_base_line/internship-causal-embedding/finetune_eval/results/followupqg/checkpoint_best.pt


In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

ckpt = torch.load(CKPT_PATH, map_location=device)
cfg  = ckpt["cfg"]

backbone = cfg.get("backbone", "google-bert/bert-base-uncased")
pooling  = cfg.get("pooling",  "cls")
max_len  = cfg.get("max_seq_length", 256)

print(f"Backbone : {backbone}")
print(f"Pooling  : {pooling}")
print(f"Max len  : {max_len}")
print(f"Val MRR  : {ckpt['metrics']['mrr']:.4f}  (best checkpoint)")

Device: cuda


Backbone : google-bert/bert-base-uncased
Pooling  : cls
Max len  : 256
Val MRR  : 0.6981  (best checkpoint)


In [12]:
tokenizer = AutoTokenizer.from_pretrained(backbone)

model = BiEncoder(
    model_name=backbone,
    pooling_strategy=pooling,
    anchor_prefix="",
    positive_prefix="",
).to(device)

model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print("Model loaded and ready.")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9567.80it/s]
[transformers] BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10471.03it/s]
[transformers] BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                 

Model loaded and ready.


## 2 — Scoring helper

In [13]:
@torch.no_grad()
def score(sentence_a: str, sentence_b: str) -> float:
    """Return a similarity score in [0, 1] for the (A → B) direction.

    A is treated as the anchor (cause / preceding context).
    B is treated as the positive (effect / follow-up).
    """
    def tokenize(text):
        enc = tokenizer(
            [text],
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors="pt",
        )
        return enc["input_ids"].to(device), enc["attention_mask"].to(device)

    a_ids, a_mask = tokenize(sentence_a)
    b_ids, b_mask = tokenize(sentence_b)

    a_emb = model.encode_anchor(a_ids, a_mask)    # (1, D) L2-normalised
    b_emb = model.encode_positive(b_ids, b_mask)  # (1, D) L2-normalised

    cosine = (a_emb * b_emb).sum().item()          # in [-1, 1]
    return (cosine + 1.0) / 2.0                    # scale to [0, 1]

print("score() ready.")

score() ready.


## 3 — Test your own sentences

Edit `SENTENCE_A` and `SENTENCE_B` and re-run the cell.

In [15]:
# ── CHANGE ME ─────────────────────────────────────────────────────────────────
SENTENCE_A = "The server ran out of disk space."
SENTENCE_B = "The application crashed and logs were lost."
# ──────────────────────────────────────────────────────────────────────────────

s = score(SENTENCE_A, SENTENCE_B)

print(f"Sentence A : {SENTENCE_A}")
print(f"Sentence B : {SENTENCE_B}")
print()
print(f"Score      : {s:.4f}  (0 = unrelated, 1 = strong causal/follow-up)")

# Simple label for quick reading
if s >= 0.75:
    label = "STRONG match"
elif s >= 0.60:
    label = "Moderate match"
elif s >= 0.45:
    label = "Weak match"
else:
    label = "Likely unrelated"
print(f"Label      : {label}")

Sentence A : The server ran out of disk space.
Sentence B : The application crashed and logs were lost.

Score      : 0.7444  (0 = unrelated, 1 = strong causal/follow-up)
Label      : Moderate match


## 4 — Batch comparison (optional)

Score multiple candidate B sentences against a single anchor A.

In [ ]:
ANCHOR = "The server ran out of disk space."

CANDIDATES = [
    "The application crashed and logs were lost.",
    "The team decided to order pizza for lunch.",
    "Disk usage monitoring alerts were triggered.",
    "The database failed to write new records.",
    "Users complained about slow response times.",
]

results = [(c, score(ANCHOR, c)) for c in CANDIDATES]
results.sort(key=lambda x: x[1], reverse=True)

print(f"Anchor: {ANCHOR}\n")
print(f"{'Score':>7}  Candidate")
print("-" * 80)
for cand, s in results:
    print(f"  {s:.4f}  {cand}")

## 5 — Compare all available checkpoints on the same pair

In [ ]:
SENTENCE_A = "The server ran out of disk space."
SENTENCE_B = "The application crashed and logs were lost."

DATASETS = ["aep_causal", "followupqg", "multiwoz_v24", "qrecc", "workflow"]

print(f"A: {SENTENCE_A}")
print(f"B: {SENTENCE_B}\n")
print(f"{'Dataset':<30} {'Variant':<15} {'Val MRR':>8}  {'Score':>7}")
print("-" * 70)

for ds in DATASETS:
    for variant in ["", "_no_inbatch"]:
        ckpt_path = RESULTS_DIR / (ds + variant) / "checkpoint_best.pt"
        if not ckpt_path.exists():
            continue
        c = torch.load(ckpt_path, map_location=device)
        cfg_c = c["cfg"]
        m = BiEncoder(
            model_name=cfg_c.get("backbone", "google-bert/bert-base-uncased"),
            pooling_strategy=cfg_c.get("pooling", "cls"),
        ).to(device)
        m.load_state_dict(c["model_state_dict"])
        m.eval()

        tok_c = AutoTokenizer.from_pretrained(cfg_c.get("backbone", "google-bert/bert-base-uncased"))
        ml = cfg_c.get("max_seq_length", 256)

        @torch.no_grad()
        def _score(a_txt, b_txt, _m=m, _tok=tok_c, _ml=ml):
            def tok(t):
                e = _tok([t], padding=True, truncation=True, max_length=_ml, return_tensors="pt")
                return e["input_ids"].to(device), e["attention_mask"].to(device)
            ai, am = tok(a_txt)
            bi, bm = tok(b_txt)
            a_emb = _m.encode_anchor(ai, am)
            b_emb = _m.encode_positive(bi, bm)
            cos = (a_emb * b_emb).sum().item()
            return (cos + 1.0) / 2.0

        s = _score(SENTENCE_A, SENTENCE_B)
        val_mrr = c["metrics"]["mrr"]
        variant_label = "no_inbatch" if variant else "in_batch"
        print(f"{ds:<30} {variant_label:<15} {val_mrr:>8.4f}  {s:>7.4f}")